In [1]:
import collections, random, numpy as np, torch
import evaluate as eval_lib
from datasets import load_dataset
from tqdm import tqdm
from transformers import DistilBertTokenizerFast, DistilBertForQuestionAnswering

MODEL_DIR = "../distilbert_squad2_finetuned/checkpoint-69000"
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_DIR)
model     = DistilBertForQuestionAnswering.from_pretrained(MODEL_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); model.to(device)

def prepare_features(ex,max_len=384,doc_stride=128):
    pr=tokenizer.padding_side=="right"
    tok=tokenizer(ex["question" if pr else "context"],ex["context" if pr else "question"],
                  truncation="longest_first",max_length=max_len,stride=doc_stride,
                  return_overflowing_tokens=True,return_offsets_mapping=True,padding="max_length")
    mp=tok.pop("overflow_to_sample_mapping"); tok["example_id"]=[ex["id"][i] for i in mp]; return tok
def postprocess(p,f,e):  # identical logic as above
    s,e_=p; per=collections.defaultdict(list)
    [per[f["example_id"][i]].append(i) for i in range(len(f["example_id"]))]
    out=collections.OrderedDict(); n_best,max_len=20,30
    for ex in e:
        cand=[]
        for fi in per[ex["id"]]:
            offs=f["offset_mapping"][fi]
            for si in np.argsort(s[fi])[-n_best:]:
                for ei in np.argsort(e_[fi])[-n_best:]:
                    if ei<si or ei-si+1>max_len or offs[si] is None or offs[ei] is None: continue
                    cand.append({"score":s[fi][si]+e_[fi][ei],"start":offs[si][0],"end":offs[ei][1]})
        out[ex["id"]]={"text":ex["context"][cand[-1]["start"]:cand[-1]["end"]],"score":cand[-1]["score"]} if cand else {"text":"","score":0}
    return out

examples=load_dataset("squad_v2")["validation"]; features=prepare_features(examples); bs=8
all_s,all_e=[],[]
for i in tqdm(range(0,len(features["input_ids"]),bs)):
    b={k:torch.tensor(v[i:i+bs]).to(device) for k,v in features.items() if k in ["input_ids","attention_mask"]}
    with torch.no_grad(): o=model(**b)
    all_s.extend(o.start_logits.cpu().numpy()); all_e.extend(o.end_logits.cpu().numpy())
print("Inference done – switch to Cell 2.")


/Users/seanhall/Desktop/NLPFinalProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1517/1517 [16:47<00:00,  1.51it/s]

Inference done – switch to Cell 2.


In [2]:
preds=postprocess((all_s,all_e),features,examples)
metric=eval_lib.load("squad_v2")
refs=[{"id":ex["id"],"answers":ex["answers"]} for ex in examples]
pl=[{"id":ex["id"],"prediction_text":preds[ex["id"]]["text"],
     "no_answer_probability":1.0 if preds[ex["id"]]["text"]=="" else 0.0} for ex in examples]
r=metric.compute(predictions=pl,references=refs)
print(f"EM {r['exact']:.2f} | F1 {r['f1']:.2f}")

wrong = [ {"q":ex["question"], "p":preds[ex["id"]]["text"],
           "g":ex["answers"]["text"][0] if ex["answers"]["text"] else "No answer"}
          for ex in examples
          if (ex['answers']['text'] and preds[ex['id']]['text'] not in ex['answers']['text'])
          or (not ex['answers']['text'] and preds[ex['id']]['text']!="") ]

for i,s in enumerate(random.sample(wrong, min(3,len(wrong))),1):
    print(f"\nEx {i}\nQ: {s['q']}\nPred: '{s['p']}'\nGold: '{s['g']}'")

EM 66.27 | F1 70.08

Ex 1
Q: Where were servers hosted? 
Pred: 'thousands of large companies, educational institutions, and government agencies'
Gold: 'No answer'

Ex 2
Q: What faults other than the San Andreas can produce a magnitude 8.0 event? 
Pred: 'San Jacinto Fault'
Gold: 'No answer'

Ex 3
Q:  What did the Gulf War do on purpose in the early 1990s?
Pred: 'brought several hundred thousand US and allied non-Muslim military personnel to Saudi Arabian soil to put an end to Saddam Hussein's occupation of Kuwait'
Gold: 'No answer'
